In [3]:
import comfy.cli_args

comfy.cli_args.args.disable_xformers = True
comfy.cli_args.args.lowvram = True
#comfy.cli_args.args.fp16_unet=False
#comfy.cli_args.args.fp16_vae=False
#comfy.cli_args.args.fp16_text_enc=False
import comfy.sd
import comfy.utils
import torch
import node_helpers
import comfy.model_management as mm

torch.inference_mode();

In [5]:
unet_path = "/workspace/ComfyUI/models/diffusion_models/wan2.1_vace_14B_fp8_e4m3fn.safetensors"
clip_path = "/workspace/ComfyUI/models/text_encoders/umt5_xxl_fp16.safetensors"
vae_path = "/workspace/ComfyUI/models/vae/wan_2.1_vae.safetensors"
loras = [
    ("/workspace/ComfyUI/models/loras/wan21t2v/Wan21_CausVid_14B_T2V_lora_rank32_v2.safetensors", 0.5, 1.0),
    ("/workspace/ComfyUI/models/loras/moni/adapter_model_2048_e30.safetensors", 1.0, 1.0),
]

In [6]:
try:
    del model
except:
    pass
try:
    del clip
except:
    pass
# reload_py_files("evc")
clip = comfy.sd.load_clip([clip_path], clip_type=comfy.sd.CLIPType.WAN)
model = evc.sd.load_diffusion_model(unet_path, model_options={})
for lora_path, strength_model, strength_clip in loras:
    lora = comfy.utils.load_torch_file(lora_path, safe_load=True)
    model, clip = comfy.sd.load_lora_for_models(model, clip, lora, strength_model, strength_clip)


In [7]:
if False:
    pos_text = "m0n1c4xxx close-up portrait of a 45 year old mexican woman smiling. 4k HD"
    neg_text = "low_quality"
    with torch.inference_mode():
        pos_tokens = clip.encode_from_tokens_scheduled(clip.tokenize(pos_text))
        neg_tokens = clip.encode_from_tokens_scheduled(clip.tokenize(neg_text))
    del clip
    torch.save([pos_tokens, neg_tokens], "posneg.pt")
else:
    pos_tokens, neg_tokens = torch.load("posneg.pt")

In [9]:

if True:
    vae = comfy.sd.VAE(comfy.utils.load_torch_file(vae_path))


    def wan_latent(positive, negative, vae, width, height, length, batch_size, strength):
        latent_length = ((length - 1) // 4) + 1
        control_video = torch.ones((length, height, width, 3)) * 0.5 - 0.5
        mask = torch.ones((length, height, width, 1))

        inactive = (control_video * (1 - mask)) + 0.5
        reactive = torch.ones((length, height, width, 3)) * 0.5  #(control_video * mask) + 0.5

        inactive = vae.encode(inactive[:, :, :, :3])
        reactive = vae.encode(reactive[:, :, :, :3])

        control_video_latent = torch.cat((inactive, reactive), dim=1)

        vae_stride = 8
        height_mask = height // vae_stride
        width_mask = width // vae_stride
        mask = mask.view(length, height_mask, vae_stride, width_mask, vae_stride)
        mask = mask.permute(2, 4, 0, 1, 3)
        mask = mask.reshape(vae_stride * vae_stride, length, height_mask, width_mask)
        mask = torch.nn.functional.interpolate(mask.unsqueeze(0), size=(latent_length, height_mask, width_mask),
                                               mode='nearest-exact').squeeze(0)

        mask = mask.unsqueeze(0)

        positive = node_helpers.conditioning_set_values(positive,
                                                        {"vace_frames": [control_video_latent], "vace_mask": [mask],
                                                         "vace_strength": [strength]}, append=True)
        negative = node_helpers.conditioning_set_values(negative,
                                                        {"vace_frames": [control_video_latent], "vace_mask": [mask],
                                                         "vace_strength": [strength]}, append=True)

        latent = torch.zeros([batch_size, 16, latent_length, height // 8, width // 8],
                             device=comfy.model_management.intermediate_device())
        out_latent = {"samples": latent}

        return positive, negative, out_latent


    with torch.inference_mode():
        positive, negative, out_latent = wan_latent(
            pos_tokens, neg_tokens, vae, width=720, height=1280, length=1, batch_size=1, strength=1.0)
    torch.save([positive, negative, out_latent], "inputs.pt")
else:
    positive, negative, out_latent = torch.load("inputs.pt")


In [7]:
import importlib
import pathlib
import sys
import os


def reload_py_files(folder_path):
    for name in list(sys.modules.keys()):
        if name.startswith("comfy."):
            del sys.modules[name]

    """Reload all .py modules from a folder that have been imported."""
    folder = pathlib.Path(folder_path).resolve()

    # Ensure folder is in sys.path so imports work
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

    for py_file in folder.glob("*.py"):
        module_name = py_file.stem
        if module_name in sys.modules:
            print(f"Reloading {module_name}")
            importlib.reload(sys.modules[module_name])
        else:
            print(f"Importing {module_name}")
            importlib.import_module(module_name)

In [2]:
%load_ext autoreload
%aimport evc.sd
%aimport evc.sample
%aimport evc.samplers
%aimport evc.model_base
%aimport evc.model_detection
%aimport evc.supported_models
%aimport evc.supported_models_base


In [9]:
%%coverage

try:
    del model
except:
    pass
# reload_py_files("evc")
model = evc.sd.load_diffusion_model(unet_path, model_options={})

UsageError: Cell magic `%%coverage` not found.


In [8]:
from IPython.core.magic import register_cell_magic
import coverage as cover


@register_cell_magic
def coverage(line, cell):
    cov = cover.Coverage(
        source=["evc"],  # or ["my_module"] if you want to narrow it down
        data_file=".coverage_ipy",
        auto_data=True
    )
    cov.start()
    try:
        exec(cell, globals())
    finally:
        cov.stop()
        cov.save()
        # cov.html_report(directory="coverage_")
        cov.xml_report(outfile="coverage.xml")
        # cov.report()

In [13]:
import evc.sample

seed = 999989
with torch.inference_mode():
    latent_image = out_latent["samples"]
    noise = torch.randn(latent_image.size(), dtype=latent_image.dtype, layout=latent_image.layout,
                        generator=torch.manual_seed(seed), device="cpu")

    samples = evc.sample.sample(
        model=model,
        noise=noise,
        seed=seed,
        steps=4,
        cfg=1.0,
        sampler_name="uni_pc",
        scheduler="simple",
        positive=positive,
        negative=negative,
        latent_image=latent_image,
        denoise=1.0)
    # output = nodes.common_ksampler(model=model, 
    #                 seed=0, 
    #                 steps=20,
    #                 cfg=3.0, 
    #                 sampler_name="uni_pc", 
    #                 scheduler="simplde", 
    #                 positive=positive,
    #                 negative=negative,
    #                 latent=out_latent, 
    #                 denoise=1.0)


  0%|          | 0/4 [00:00<?, ?it/s]

In [16]:
type(model.model)
#type(model).mro()

comfy.model_base.WAN21_Vace

In [8]:
vae = comfy.sd.VAE(comfy.utils.load_torch_file(vae_path))

In [16]:
with torch.inference_mode():
    images = vae.decode(samples)
    images = images.reshape(-1, images.shape[-3], images.shape[-2], images.shape[-1])

In [17]:
from fractions import Fraction
from comfy_api.input_impl import VideoFromComponents
from comfy_api.util import VideoCodec, VideoComponents, VideoContainer

video = VideoFromComponents(VideoComponents(images=images, frame_rate=Fraction(16)))
video.save_to(
    "vid3.mp4",
    # format=format,
    # codec=codec,
    # metadata=saved_metadata
)

In [22]:
import PIL.Image as Image
import numpy as np
img = Image.fromarray(np.clip(images[0].detach().numpy() * 255, 0, 255).astype(np.uint8))
img.save("vid3.jpg")


In [ ]:
print(f"Allocated: {torch.cuda.memory_allocated() / 1024 ** 2:.1f} MB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024 ** 2:.1f} MB")

In [23]:
torch.cuda.empty_cache()

In [26]:
import kes
kes.vace(
    out_path = "vid5.mp4",
)

  0%|          | 0/4 [00:00<?, ?it/s]